# CARE autoencoder training on Colab (Farm C)

Trains the per-dataset autoencoders of `docs/care-autoencoder-protocol.md` for Wind
Farm C, the held-out confirmation farm. It only trains and saves scores; the CARE
score is computed locally with the decision rule already frozen on Farms A and B.

1. **Runtime → Change runtime type → T4 GPU** (CPU also works, just slower).
2. Run all cells. Upload the input ZIP from `scripts/package_care_colab.py` when asked.
3. Allow Google Drive access when asked. Results go to
   `MyDrive/HydroPredict/care-autoencoder-v1-C`; rerunning resumes where it stopped.

The CARE data (5.5 GB) downloads straight from Zenodo and is checked against its
published MD5. Most of the run time is reading the large CSV files, not the GPU.

In [ ]:
import hashlib
import json
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly the CARE Colab input ZIP.")
WORK = Path(tempfile.mkdtemp(prefix="care-"))
INPUT = WORK / "inputs"


def extract_owned_zip(data, destination):
    destination.mkdir(parents=True, exist_ok=True)
    path = destination.with_suffix(".zip")
    path.write_bytes(data)
    with zipfile.ZipFile(path) as bundle:
        for entry in bundle.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)


extract_owned_zip(next(iter(uploaded.values())), INPUT)
manifest = json.loads((INPUT / "input_manifest.json").read_text())
for name, expected in manifest["files_sha256"].items():
    if hashlib.sha256((INPUT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Input file changed: {name}")
PROJECT = WORK / "project"
extract_owned_zip((INPUT / "project.zip").read_bytes(), PROJECT)
print("Source commit:", manifest["source_commit"])

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)

In [ ]:
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("PyTorch:", torch.__version__, "| GPU:", GPU or "none (CPU run)")

## Download and verify CARE

The archive is saved on the Colab disk, not Drive. If the runtime restarts, this
cell downloads it again. Only the farms this run needs are extracted.

In [ ]:
DATA_ZIP = Path("/content/CARE_To_Compare.zip")
if not DATA_ZIP.exists() or DATA_ZIP.stat().st_size != manifest["zenodo_bytes"]:
    subprocess.run(["wget", "-q", "-c", "-O", str(DATA_ZIP), manifest["zenodo_url"]], check=True)
md5 = hashlib.md5()
with DATA_ZIP.open("rb") as stream:
    for block in iter(lambda: stream.read(16 * 1024 * 1024), b""):
        md5.update(block)
if md5.hexdigest() != manifest["zenodo_md5"]:
    raise ValueError("Downloaded CARE archive does not match the published MD5.")
DATA = Path("/content/care")
farms = [manifest["parity"]["farm"], *manifest["train_farms"]]
with zipfile.ZipFile(DATA_ZIP) as bundle:
    members = [m for m in bundle.namelist() if any(f"/{farm}/" in m for farm in farms)]
    bundle.extractall(DATA, members)
ROOT = DATA / "CARE_To_Compare"
print("Verified and extracted:", farms)

## Device parity check

Retrains one Farm A dataset on this runtime and compares it with the laptop's scores.
GPU arithmetic differs slightly from CPU, so small differences are expected; a weak
correlation means the environment does not reproduce the local model and the run stops.

In [ ]:
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
RUN_ROOT = Path("/content/drive/MyDrive/HydroPredict/care-autoencoder-v1-C")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
parity = manifest["parity"]
PARITY_DIR = WORK / "parity"
subprocess.run(
    [sys.executable, "-m", "scripts.train_care_autoencoder", str(ROOT)]
    + ["--farms", parity["farm"], "--events", str(parity["event_id"])]
    + ["--scores", str(PARITY_DIR), "--output", str(PARITY_DIR), "--device", "auto"],
    cwd=PROJECT,
    check=True,
)
name = f"{parity['farm'].replace(' ', '_')}_{parity['event_id']}.npz"
ours = np.load(PARITY_DIR / name)["scores"]
local = np.load(INPUT / "parity_reference.npz")["scores"]
both = ~np.isnan(ours) & ~np.isnan(local)
result = dict(
    gpu=GPU,
    unscored_match=bool((np.isnan(ours) == np.isnan(local)).all()),
    correlation=float(np.corrcoef(ours[both], local[both])[0, 1]),
    median_relative_difference=float(np.median(np.abs(ours[both] - local[both]) / local[both])),
)
(RUN_ROOT / "parity.json").write_text(json.dumps(result, indent=2))
print(result)
if not result["unscored_match"] or result["correlation"] < 0.95:
    raise RuntimeError("This runtime does not reproduce the local model closely enough.")

## Train Farm C

About 58 datasets. Progress is saved to Drive after each one; if Colab disconnects,
rerun every cell and training continues from the next unfinished dataset.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "scripts.train_care_autoencoder", str(ROOT), "--farms"]
    + manifest["train_farms"]
    + ["--scores", str(RUN_ROOT / "scores"), "--output", str(RUN_ROOT), "--device", "auto"],
    cwd=PROJECT,
    check=True,
)
audit = json.loads((RUN_ROOT / "training.json").read_text())
(RUN_ROOT / "source_commit.txt").write_text(manifest["source_commit"])
print(len(audit), "datasets trained; total minutes:", round(sum(a["seconds"] for a in audit) / 60))

## Bring the results back

Download the ZIP below and tell Claude where it is saved. It contains score files,
model weights, `training.json`, `parity.json` and the run configuration. Drive keeps
the persistent copy.

In [ ]:
import shutil

export = shutil.make_archive("/content/care-autoencoder-v1-C", "zip", root_dir=RUN_ROOT)
files.download(export)